# 🏷️ **PDF Page Importance Labeling Tool**

## **Purpose:**
Create a labeled dataset for training a CNN to classify pages as:
- ✅ **Important** (content pages)
- ❌ **Not Important** (cover, TOC, etc.)

## **Features:**
- 📸 Visual page display
- ⌨️ Keyboard shortcuts (I = Important, N = Not Important)
- ⏭️ Skip rest of PDF button
- 🎯 Smart sampling (balanced dataset)
- 💾 Auto-save to Google Drive
- 📊 Real-time statistics

---

## 🔧 **1. Setup & Installation**

In [1]:
# Install dependencies
!pip install -q PyMuPDF pillow ipywidgets

print("✅ Dependencies installed!")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.1/24.1 MB 23.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 30.7 MB/s eta 0:00:00
✅ Dependencies installed!


## 📂 **2. Mount Google Drive & Configure Paths**

In [2]:
from google.colab import drive
import os
from pathlib import Path

# Mount Google Drive
if not os.path.exists('/content/drive/MyDrive'):
    drive.mount('/content/drive')
    print("✅ Google Drive mounted!")
else:
    print("✅ Google Drive already mounted!")

# Configure paths
BASE_DIR = Path("/content/drive/Othercomputers/My Mac/Multi-lingual-Buddhist-Conversational-Agent/")
PDF_FOLDER = BASE_DIR / "data" / "raw" / "old_books"
PAGE_CLASSIFIER_DATA = BASE_DIR / "data" / "page_classifier_data"

# Create directories
IMAGES_DIR = PAGE_CLASSIFIER_DATA / "images"
LABELS_DIR = PAGE_CLASSIFIER_DATA / "labels"

for directory in [PAGE_CLASSIFIER_DATA, IMAGES_DIR, LABELS_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

print(f"\n📂 Directory Structure:")
print(f"   PDFs: {PDF_FOLDER}")
print(f"   Images: {IMAGES_DIR}")
print(f"   Labels: {LABELS_DIR}")

# Check if PDFs exist
pdf_files = list(PDF_FOLDER.glob("*.pdf"))
print(f"\n📚 Found {len(pdf_files)} PDF files")

Mounted at /content/drive
✅ Google Drive mounted!

📂 Directory Structure:
   PDFs: /content/drive/Othercomputers/My Mac/Multi-lingual-Buddhist-Conversational-Agent/data/raw/old_books
   Images: /content/drive/Othercomputers/My Mac/Multi-lingual-Buddhist-Conversational-Agent/data/page_classifier_data/images
   Labels: /content/drive/Othercomputers/My Mac/Multi-lingual-Buddhist-Conversational-Agent/data/page_classifier_data/labels

📚 Found 0 PDF files


## 🎨 **3. Labeling Tool - Main Interface**

In [3]:
import fitz  # PyMuPDF
from PIL import Image
import json
import io
from datetime import datetime
from IPython.display import display, HTML, clear_output
import ipywidgets as widgets
from collections import defaultdict

class LabelingSession:
    def __init__(self):
        self.current_pdf_idx = 0
        self.current_page_idx = 0
        self.pdf_files = sorted(list(PDF_FOLDER.rglob("*.pdf")))
        self.current_doc = None
        self.labels = {}  # {pdf_name: {page_num: label}}
        self.session_stats = defaultdict(lambda: {'important': 0, 'not_important': 0})
        self.load_existing_labels()

    def load_existing_labels(self):
        """Load previously saved labels"""
        labels_file = LABELS_DIR / "labels.json"
        if labels_file.exists():
            with open(labels_file, 'r') as f:
                self.labels = json.load(f)
            print(f"✅ Loaded existing labels: {sum(len(v) for v in self.labels.values())} pages labeled")
        else:
            print("📝 Starting fresh labeling session")

    def save_labels(self):
        """Save labels to JSON"""
        labels_file = LABELS_DIR / "labels.json"
        with open(labels_file, 'w') as f:
            json.dump(self.labels, f, indent=2)
        print(f"💾 Labels saved: {sum(len(v) for v in self.labels.values())} total pages labeled")

    def get_current_pdf_name(self):
        if self.current_pdf_idx < len(self.pdf_files):
            return self.pdf_files[self.current_pdf_idx].stem
        return None

    def load_pdf(self, pdf_idx):
        """Load a PDF document"""
        if self.current_doc:
            self.current_doc.close()

        self.current_pdf_idx = pdf_idx
        self.current_page_idx = 0
        pdf_path = self.pdf_files[pdf_idx]
        self.current_doc = fitz.open(pdf_path)

        # Initialize labels dict for this PDF if not exists
        pdf_name = pdf_path.stem
        if pdf_name not in self.labels:
            self.labels[pdf_name] = {}

    def get_page_image(self, page_num):
        """Extract page as PIL Image"""
        page = self.current_doc[page_num]
        pix = page.get_pixmap(dpi=150)
        img = Image.frombytes("RGB", [pix.width, pix.height], pix.samples)
        return img

    def save_page_image(self, page_num, label):
        """Save page image directly to label folder with simple numbering"""
        img = self.get_page_image(page_num)

        # Create label folder
        label_folder = IMAGES_DIR / label
        label_folder.mkdir(parents=True, exist_ok=True)

        # Find next available number across all PDFs
        existing_files = list(label_folder.glob("page_*.png"))
        if existing_files:
            # Extract numbers from filenames
            numbers = []
            for f in existing_files:
                try:
                    num = int(f.stem.split('_')[1])
                    numbers.append(num)
                except:
                    pass
            next_num = max(numbers) + 1 if numbers else 0
        else:
            next_num = 0

        # Save image with simple incremental name
        img_filename = f"page_{next_num:04d}.png"
        img_path = label_folder / img_filename
        img.save(img_path)

        return str(img_path)

    def label_page(self, label):
        """Label current page"""
        pdf_name = self.get_current_pdf_name()
        page_num = self.current_page_idx

        # Save image
        img_path = self.save_page_image(page_num, label)

        # Store label
        self.labels[pdf_name][str(page_num)] = {
            'label': label,
            'image_path': img_path,
            'timestamp': datetime.now().isoformat()
        }

        # Update stats
        self.session_stats[pdf_name][label] += 1

        # Auto-save every 5 labels
        total_labeled = sum(len(v) for v in self.labels.values())
        if total_labeled % 5 == 0:
            self.save_labels()

    def skip_rest_of_pdf(self, default_label='important'):
        """Skip rest of current PDF with default label"""
        pdf_name = self.get_current_pdf_name()
        total_pages = len(self.current_doc)
        skipped = 0

        # Label remaining pages
        for page_num in range(self.current_page_idx, total_pages):
            if str(page_num) not in self.labels[pdf_name]:
                # Don't save images, just mark as labeled
                self.labels[pdf_name][str(page_num)] = {
                    'label': default_label,
                    'image_path': None,  # Not saved
                    'timestamp': datetime.now().isoformat(),
                    'skipped': True
                }
                skipped += 1

        self.save_labels()
        return skipped

    def get_stats(self):
        """Get labeling statistics"""
        stats = {
            'total_pdfs': len(self.pdf_files),
            'current_pdf': self.current_pdf_idx + 1,
            'total_pages_labeled': sum(len(v) for v in self.labels.values()),
            'important': 0,
            'not_important': 0,
            'images_saved': 0
        }

        for pdf_labels in self.labels.values():
            for page_data in pdf_labels.values():
                if page_data['label'] == 'important':
                    stats['important'] += 1
                else:
                    stats['not_important'] += 1

                if page_data.get('image_path'):
                    stats['images_saved'] += 1

        return stats

# Initialize session
session = LabelingSession()

if len(session.pdf_files) == 0:
    print("❌ No PDF files found! Please check the PDF_FOLDER path.")
else:
    print(f"✅ Labeling session initialized with {len(session.pdf_files)} PDFs")

📝 Starting fresh labeling session
✅ Labeling session initialized with 28 PDFs


## 🖼️ **4. Interactive Labeling Interface**

In [4]:
# ═══════════════════════════════════════════════════════
# INTERACTIVE WIDGET-BASED INTERFACE
# ═══════════════════════════════════════════════════════

# Output widget for displaying page image
image_output = widgets.Output()
stats_output = widgets.Output()
info_output = widgets.Output()

# Buttons
btn_important = widgets.Button(
    description='✅ IMPORTANT (I)',
    button_style='success',
    layout=widgets.Layout(width='200px', height='50px')
)

btn_not_important = widgets.Button(
    description='❌ NOT IMPORTANT (N)',
    button_style='danger',
    layout=widgets.Layout(width='200px', height='50px')
)

btn_skip_pdf = widgets.Button(
    description='⏭️ SKIP REST OF PDF',
    button_style='warning',
    layout=widgets.Layout(width='200px', height='50px')
)

btn_prev = widgets.Button(
    description='← Previous',
    button_style='info',
    layout=widgets.Layout(width='150px')
)

btn_next_pdf = widgets.Button(
    description='Next PDF →',
    button_style='info',
    layout=widgets.Layout(width='150px')
)

# ═══════════════════════════════════════════════════════
# DISPLAY FUNCTIONS
# ═══════════════════════════════════════════════════════

def update_display():
    """Update the page display"""
    if session.current_doc is None:
        return

    # Clear outputs
    image_output.clear_output(wait=True)
    info_output.clear_output(wait=True)
    stats_output.clear_output(wait=True)

    # Display current page info
    with info_output:
        pdf_name = session.get_current_pdf_name()
        total_pages = len(session.current_doc)
        page_num = session.current_page_idx

        # Check if already labeled
        already_labeled = str(page_num) in session.labels.get(pdf_name, {})
        label_status = ""
        if already_labeled:
            current_label = session.labels[pdf_name][str(page_num)]['label']
            label_status = f"<span style='color: orange;'>⚠️ Already labeled as: <b>{current_label}</b></span>"

        display(HTML(f"""
        <div style='background: #f0f0f0; padding: 15px; border-radius: 10px; margin-bottom: 10px;'>
            <h3>📄 {pdf_name}</h3>
            <p><b>Page:</b> {page_num + 1} / {total_pages} | <b>PDF:</b> {session.current_pdf_idx + 1} / {len(session.pdf_files)}</p>
            {label_status}
        </div>
        """))

    # Display page image
    with image_output:
        img = session.get_page_image(session.current_page_idx)
        # Resize for display (max width 800px)
        display_img = img.copy()
        if display_img.width > 800:
            ratio = 800 / display_img.width
            new_height = int(display_img.height * ratio)
            display_img = display_img.resize((800, new_height))
        display(display_img)

    # Display statistics
    with stats_output:
        stats = session.get_stats()
        total = stats['important'] + stats['not_important']
        balance = abs(stats['important'] - stats['not_important'])

        balance_msg = "✅ Balanced" if balance <= 5 else f"⚠️ Imbalanced (+{balance})"

        display(HTML(f"""
        <div style='background: #e8f5e9; padding: 15px; border-radius: 10px;'>
            <h4>📊 Session Statistics</h4>
            <p><b>Total Labeled:</b> {total} pages | <b>Images Saved:</b> {stats['images_saved']}</p>
            <p>✅ <b>Important:</b> {stats['important']} | ❌ <b>Not Important:</b> {stats['not_important']}</p>
            <p><b>Balance:</b> {balance_msg}</p>
        </div>
        """))

# ═══════════════════════════════════════════════════════
# BUTTON CALLBACKS
# ═══════════════════════════════════════════════════════

def on_important_clicked(b):
    session.label_page('important')
    # Move to next page
    session.current_page_idx += 1
    if session.current_page_idx >= len(session.current_doc):
        # Move to next PDF
        session.current_pdf_idx += 1
        if session.current_pdf_idx < len(session.pdf_files):
            session.load_pdf(session.current_pdf_idx)
        else:
            with info_output:
                clear_output()
                print("🎉 All PDFs completed!")
            return
    update_display()

def on_not_important_clicked(b):
    session.label_page('not_important')
    # Move to next page
    session.current_page_idx += 1
    if session.current_page_idx >= len(session.current_doc):
        # Move to next PDF
        session.current_pdf_idx += 1
        if session.current_pdf_idx < len(session.pdf_files):
            session.load_pdf(session.current_pdf_idx)
        else:
            with info_output:
                clear_output()
                print("🎉 All PDFs completed!")
            return
    update_display()

def on_skip_pdf_clicked(b):
    skipped = session.skip_rest_of_pdf(default_label='important')
    with info_output:
        print(f"⏭️ Skipped {skipped} pages (marked as 'important')")

    # Move to next PDF
    session.current_pdf_idx += 1
    if session.current_pdf_idx < len(session.pdf_files):
        session.load_pdf(session.current_pdf_idx)
        update_display()
    else:
        with info_output:
            clear_output()
            print("🎉 All PDFs completed!")

def on_prev_clicked(b):
    if session.current_page_idx > 0:
        session.current_page_idx -= 1
        update_display()

def on_next_pdf_clicked(b):
    session.current_pdf_idx += 1
    if session.current_pdf_idx < len(session.pdf_files):
        session.load_pdf(session.current_pdf_idx)
        update_display()
    else:
        with info_output:
            clear_output()
            print("🎉 All PDFs completed!")

# Attach callbacks
btn_important.on_click(on_important_clicked)
btn_not_important.on_click(on_not_important_clicked)
btn_skip_pdf.on_click(on_skip_pdf_clicked)
btn_prev.on_click(on_prev_clicked)
btn_next_pdf.on_click(on_next_pdf_clicked)

# ═══════════════════════════════════════════════════════
# KEYBOARD SHORTCUTS
# ═══════════════════════════════════════════════════════

display(HTML("""
<script>
document.addEventListener('keydown', function(event) {
    if (event.key === 'i' || event.key === 'I') {
        // Trigger Important button
        document.querySelector('button[class*="success"]').click();
    } else if (event.key === 'n' || event.key === 'N') {
        // Trigger Not Important button
        document.querySelector('button[class*="danger"]').click();
    }
});
</script>
"""))

# Layout
button_row1 = widgets.HBox([btn_important, btn_not_important, btn_skip_pdf])
button_row2 = widgets.HBox([btn_prev, btn_next_pdf])

# Display interface
display(HTML("""
<div style='background: #fff3cd; padding: 15px; border-radius: 10px; margin-bottom: 15px;'>
    <h3>⌨️ Keyboard Shortcuts</h3>
    <p><b>I</b> = Important | <b>N</b> = Not Important</p>
</div>
"""))

display(stats_output)
display(info_output)
display(button_row1)
display(button_row2)
display(image_output)

# Load first PDF and display
if len(session.pdf_files) > 0:
    session.load_pdf(0)
    update_display()
    print("\n🚀 Labeling tool ready! Use buttons or keyboard shortcuts (I/N)")
else:
    print("❌ No PDFs found to label")

Output()

Output()

Output()


🚀 Labeling tool ready! Use buttons or keyboard shortcuts (I/N)
💾 Labels saved: 5 total pages labeled
💾 Labels saved: 10 total pages labeled
💾 Labels saved: 15 total pages labeled
💾 Labels saved: 20 total pages labeled
💾 Labels saved: 25 total pages labeled
💾 Labels saved: 30 total pages labeled
💾 Labels saved: 35 total pages labeled
💾 Labels saved: 40 total pages labeled
💾 Labels saved: 45 total pages labeled
💾 Labels saved: 50 total pages labeled
💾 Labels saved: 55 total pages labeled
💾 Labels saved: 60 total pages labeled
💾 Labels saved: 65 total pages labeled
💾 Labels saved: 70 total pages labeled
💾 Labels saved: 75 total pages labeled
💾 Labels saved: 80 total pages labeled


## 📊 **5. View Statistics & Export Data**

In [5]:
# ═══════════════════════════════════════════════════════
# STATISTICS & SUMMARY
# ═══════════════════════════════════════════════════════

import pandas as pd

def generate_summary():
    """Generate detailed summary of labeling session"""

    stats = session.get_stats()

    print("="*80)
    print("📊 LABELING SESSION SUMMARY")
    print("="*80)

    print(f"\n📚 Total PDFs: {stats['total_pdfs']}")
    print(f"📄 Total Pages Labeled: {stats['total_pages_labeled']}")
    print(f"💾 Images Saved: {stats['images_saved']}")

    print(f"\n✅ Important: {stats['important']}")
    print(f"❌ Not Important: {stats['not_important']}")

    if stats['important'] + stats['not_important'] > 0:
        balance = abs(stats['important'] - stats['not_important'])
        total = stats['important'] + stats['not_important']
        print(f"\n⚖️ Class Balance: {balance} difference ({balance/total*100:.1f}%)")

    # Per-PDF breakdown
    print(f"\n📋 Per-PDF Breakdown:")
    print("-"*80)

    pdf_data = []
    for pdf_name, labels in session.labels.items():
        imp = sum(1 for v in labels.values() if v['label'] == 'important')
        not_imp = sum(1 for v in labels.values() if v['label'] == 'not_important')
        pdf_data.append({
            'PDF': pdf_name,
            'Important': imp,
            'Not Important': not_imp,
            'Total': imp + not_imp
        })

    if pdf_data:
        df = pd.DataFrame(pdf_data)
        print(df.to_string(index=False))

    print("\n" + "="*80)
    print("💾 Data saved to:")
    print(f"   Labels: {LABELS_DIR / 'labels.json'}")
    print(f"   Images: {IMAGES_DIR}")
    print("="*80)

# Generate summary
generate_summary()

📊 LABELING SESSION SUMMARY

📚 Total PDFs: 28
📄 Total Pages Labeled: 998
💾 Images Saved: 54

✅ Important: 978
❌ Not Important: 20

⚖️ Class Balance: 958 difference (96.0%)

📋 Per-PDF Breakdown:
--------------------------------------------------------------------------------
         PDF  Important  Not Important  Total
බාගතකර_ගැනීම        978             20    998

💾 Data saved to:
   Labels: /content/drive/Othercomputers/My Mac/Multi-lingual-Buddhist-Conversational-Agent/data/page_classifier_data/labels/labels.json
   Images: /content/drive/Othercomputers/My Mac/Multi-lingual-Buddhist-Conversational-Agent/data/page_classifier_data/images


## 💾 **6. Manual Save (if needed)**

In [6]:
# Manual save button
session.save_labels()
print("✅ Labels manually saved to Google Drive!")

💾 Labels saved: 998 total pages labeled
✅ Labels manually saved to Google Drive!


## 🔍 **7. Inspect Labeled Data**

In [7]:
# ═══════════════════════════════════════════════════════
# INSPECT LABELED DATA
# ═══════════════════════════════════════════════════════

def inspect_labels():
    """Show sample of labeled pages"""

    labels_file = LABELS_DIR / "labels.json"
    if not labels_file.exists():
        print("❌ No labels file found")
        return

    with open(labels_file, 'r') as f:
        labels = json.load(f)

    # Collect all labeled pages
    important_pages = []
    not_important_pages = []

    for pdf_name, pages in labels.items():
        for page_num, data in pages.items():
            if data.get('image_path'):  # Only saved images
                if data['label'] == 'important':
                    important_pages.append((pdf_name, page_num, data['image_path']))
                else:
                    not_important_pages.append((pdf_name, page_num, data['image_path']))

    print(f"✅ Found {len(important_pages)} important and {len(not_important_pages)} not important pages\n")

    # Show samples
    import random

    print("📸 Sample Important Pages:")
    for pdf, page, path in random.sample(important_pages, min(3, len(important_pages))):
        print(f"   {pdf} - Page {page}")

    print(f"\n📸 Sample Not Important Pages:")
    for pdf, page, path in random.sample(not_important_pages, min(3, len(not_important_pages))):
        print(f"   {pdf} - Page {page}")

inspect_labels()

✅ Found 34 important and 20 not important pages

📸 Sample Important Pages:
   බාගතකර_ගැනීම - Page 31
   බාගතකර_ගැනීම - Page 17
   බාගතකර_ගැනීම - Page 38

📸 Sample Not Important Pages:
   බාගතකර_ගැනීම - Page 23
   බාගතකර_ගැනීම - Page 5
   බාගතකර_ගැනීම - Page 0


---

## 📚 **Usage Guide**

### **Workflow:**
1. Run all cells to initialize the labeling tool
2. Use the interface to label pages:
   - Click **✅ IMPORTANT** for content pages
   - Click **❌ NOT IMPORTANT** for cover/TOC/etc.
   - Use keyboard shortcuts: **I** = Important, **N** = Not Important
3. Click **⏭️ SKIP REST OF PDF** when you've seen enough (e.g., after labeling first 15 pages)
4. Labels auto-save every 5 pages
5. View statistics in Section 5

### **Output Structure:**
```
page_classifier_data/
├── images/
│   ├── important/
│   │   └── BookName/
│   │       ├── page_0000.png
│   │       └── ...
│   └── not_important/
│       └── BookName/
│           ├── page_0001.png
│           └── ...
└── labels/
    └── labels.json
```

### **Smart Sampling Strategy:**
- Label **first 15 pages** of each PDF (likely non-content)
- Label **20-25 middle pages** (likely content)
- Click "Skip Rest" to auto-label remaining as 'important'
- This creates a balanced dataset efficiently!

### **Next Steps:**
After labeling, use the saved images and labels to train a CNN classifier!

---

**Version:** 1.0  
**Created for:** Buddhist PDF Page Classification Project  
**License:** MIT
